# CMX (MiT-B2) on PST900: train for 300 epochs and evaluate

**Needed:**
* a CUDA GPU with PyTorch (CUDA) installed;
* the PST900 dataset (zip or extracted folder);
* `mit_b2.pth` (ImageNet-pretrained SegFormer weights).

**Steps:**
1. Set the two paths in cell 1.
2. Choose **Run all**.

If training is interrupted, run all again: it resumes from the last checkpoint.

## 1. Settings

In [ ]:
DATASET = r'PST900_RGBT_Dataset.zip'   # PST900 zip, or the extracted folder
MIT_B2  = r'mit_b2.pth'                # pretrained MiT-B2 weights

EPOCHS     = 300
BATCH_SIZE = 4        # use 8 on a GPU with >= 20 GB memory, 2 if out of memory
WORK       = r'cmx_work'               # everything is written here

## 2. Setup: install packages, get the CMX code, prepare the data

In [ ]:
import os, sys, glob, shutil, zipfile, subprocess, torch
assert torch.cuda.is_available(), 'No CUDA GPU found (install the CUDA build of PyTorch).'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'easydict', 'timm', 'tensorboardX', 'opencv-python', 'tqdm', 'scipy'])

WORK = os.path.abspath(WORK); os.makedirs(WORK, exist_ok=True)
REPO = os.path.join(WORK, 'RGBX_Semantic_Segmentation')
LOG  = os.path.join(WORK, 'log_PST900')

# --- official CMX code ---
if not os.path.exists(REPO):
    subprocess.check_call(['git', 'clone', '-q', 'https://github.com/huaaaliu/RGBX_Semantic_Segmentation.git', REPO])

# --- small compatibility fixes (model, loss and training schedule unchanged) ---
def fix(file, old, new):
    p = os.path.join(REPO, file); s = open(p).read()
    if new not in s:
        assert old in s, f'cannot patch {file}'
        open(p, 'w').write(s.replace(old, new))
fix('train.py', 'dataloader.next()', 'next(dataloader)')
fix('train.py', '                sum_loss += loss\n', '                sum_loss += loss.item()\n')
fix('train.py', "% (loss, (sum_loss", "% (loss.item(), (sum_loss")
fix('utils/transforms.py', 'collections.Iterable', 'collections.abc.Iterable')
fix('utils/transforms.py', 'import collections\n', 'import collections\nimport collections.abc\n')
fix('engine/engine.py', "tmp['model'], is_restore=True", "tmp['model'], is_restore=self.distributed")
fix('engine/engine.py', 'tmp = torch.load(self.continue_state_object)\n',
    "tmp = torch.load(self.continue_state_object, map_location='cpu', weights_only=False)\n")
# mixed precision: faster training, less GPU memory
fix('train.py', '    optimizer.zero_grad()\n    model.train()\n',
    "    scaler = torch.amp.GradScaler('cuda')\n    optimizer.zero_grad()\n    model.train()\n")
fix('train.py', '            loss = model(imgs, modal_xs, gts)\n',
    "            with torch.autocast('cuda', dtype=torch.float16):\n                loss = model(imgs, modal_xs, gts)\n")
fix('train.py', '            loss.backward()\n            optimizer.step()\n',
    '            scaler.scale(loss).backward()\n            scaler.step(optimizer)\n            scaler.update()\n')
# keep only the 2 newest checkpoints (each is ~800 MB)
fix('engine/engine.py', '        self.save_checkpoint(current_epoch_checkpoint)\n',
    '        self.save_checkpoint(current_epoch_checkpoint)\n'
    '        import glob as _g\n'
    '        _c = sorted(_g.glob(osp.join(checkpoint_dir, "epoch-[0-9]*.pth")), key=osp.getmtime)\n'
    '        [os.remove(_p) for _p in _c[:-2]]\n')

# --- dataset: unzip if needed, then put it in the CMX layout ---
src = os.path.abspath(DATASET)
if src.endswith('.zip'):
    out = os.path.join(WORK, 'dataset')
    if not os.path.exists(out):
        print('Unzipping...'); zipfile.ZipFile(src).extractall(out)
    src = out
root = next(r for r, d, _ in os.walk(src) if 'train' in d and 'test' in d)
DATA = os.path.join(WORK, 'pst900_cmx')
for sub in ('RGB', 'Thermal', 'Label'):
    os.makedirs(os.path.join(DATA, sub), exist_ok=True)
counts = {}
for split in ('train', 'test'):
    names = []
    for f in sorted(glob.glob(os.path.join(root, split, 'rgb', '*.png'))):
        n = os.path.basename(f); new = f'{split}_{n}'
        for sub, folder in (('RGB', 'rgb'), ('Thermal', 'thermal'), ('Label', 'labels')):
            dst = os.path.join(DATA, sub, new)
            if not os.path.exists(dst):
                shutil.copy(os.path.join(root, split, folder, n), dst)
        names.append(new[:-4])
    open(os.path.join(DATA, f'{split}.txt'), 'w').write('\n'.join(names) + '\n')
    counts[split] = len(names)
print('Images:', counts, '(PST900 should be 597 train / 288 test)')

## 3. Training config (repo defaults; PST900 has 5 classes)

In [ ]:
CLASSES = ['Background', 'Fire-Extinguisher', 'Backpack', 'Hand-Drill', 'Survivor']
if not os.path.exists(os.path.join(REPO, 'config_original.py')):
    shutil.copy(os.path.join(REPO, 'config.py'), os.path.join(REPO, 'config_original.py'))
cfg = open(os.path.join(REPO, 'config_original.py')).read()

import re
def setv(key, value):
    global cfg
    if key == 'class_names':   # this list spans several lines in the original file
        cfg, n = re.subn(r'^C\.class_names = *\[.*?\]', lambda m: f'C.class_names = {value}', cfg, flags=re.M | re.S)
    else:
        cfg, n = re.subn(rf'^C\.{key} = .*$', lambda m: f'C.{key} = {value}', cfg, flags=re.M)
    assert n == 1, key

setv('dataset_name', "'PST900'")
setv('dataset_path', repr(DATA))
setv('rgb_format', "'.png'")
setv('x_root_folder', "osp.join(C.dataset_path, 'Thermal')")
setv('x_format', "'.png'")
setv('x_is_single_channel', 'True')
setv('gt_transform', 'False')                 # label 0 = Background is a real class
setv('num_train_imgs', counts['train'])
setv('num_eval_imgs', counts['test'])
setv('num_classes', 5)
setv('class_names', repr(CLASSES))
setv('pretrained_model', repr(os.path.abspath(MIT_B2)))
setv('batch_size', BATCH_SIZE)
setv('nepochs', EPOCHS)
setv('num_workers', 0 if os.name == 'nt' else 4)   # Windows needs 0 for this repo
setv('checkpoint_start_epoch', 10)
setv('checkpoint_step', 10)
setv('log_dir', repr(LOG))
open(os.path.join(REPO, 'config.py'), 'w').write(cfg)
print(f'Config written: {EPOCHS} epochs, batch size {BATCH_SIZE}, checkpoints in {LOG}')

## 4. Train (resumes automatically if interrupted)

In [ ]:
ckpts = sorted(glob.glob(os.path.join(LOG, 'checkpoint', 'epoch-[0-9]*.pth')), key=os.path.getmtime)
cmd = [sys.executable, 'train.py', '-d', '0'] + (['-c', ckpts[-1]] if ckpts else [])
print('Resuming from', ckpts[-1] if ckpts else 'scratch')
env = dict(os.environ, PYTHONUNBUFFERED='1')
proc = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, errors='replace')
os.makedirs(LOG, exist_ok=True)
with open(os.path.join(LOG, 'train_output.txt'), 'a') as logf:
    for line in proc.stdout:
        logf.write(line)
        print(line.strip()[:200], end='\r' if 'Iter' in line else '\n')
proc.wait()
print('\nFinished, exit code', proc.returncode)

## 5. Evaluate on the test set
Computes mIoU, Dice, Sensitivity and Specificity using sliding-window inference, the same protocol as the repo.
Results are saved to `cmx_work/results.csv`.

In [ ]:
import cv2, numpy as np, pandas as pd, torch.nn as nn, torch.nn.functional as F
sys.path.insert(0, REPO); os.chdir(REPO)
from config import config
from models.builder import EncoderDecoder

ck = sorted(glob.glob(os.path.join(LOG, 'checkpoint', 'epoch-[0-9]*.pth')), key=os.path.getmtime)[-1]
model = EncoderDecoder(cfg=config, criterion=None, norm_layer=nn.BatchNorm2d)
model.load_state_dict(torch.load(ck, map_location='cpu', weights_only=False)['model'])
model = model.cuda().eval()
print('Evaluating', os.path.basename(ck))

mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
def prep(img):   # same preprocessing as the repo's loader
    return torch.from_numpy(((img / 255.0 - mean) / std).transpose(2, 0, 1)).float()[None].cuda()

@torch.no_grad()
def predict(rgb, th, crop=(480, 640), stride=2/3):
    H, W = rgb.shape[-2:]; ch, cw = crop
    sh, sw = int(np.ceil(ch * stride)), int(np.ceil(cw * stride))
    ys = sorted(set(list(range(0, max(H - ch, 0) + 1, sh)) + [max(H - ch, 0)]))
    xs = sorted(set(list(range(0, max(W - cw, 0) + 1, sw)) + [max(W - cw, 0)]))
    out = torch.zeros(1, 5, H, W, device='cuda')
    for y in ys:
        for x in xs:
            out[..., y:y+ch, x:x+cw] += model(rgb[..., y:y+ch, x:x+cw], th[..., y:y+ch, x:x+cw]).softmax(1)
    return out.argmax(1)[0].cpu().numpy()

cm = np.zeros((5, 5), np.int64)
names = open(os.path.join(DATA, 'test.txt')).read().split()
for i, n in enumerate(names):
    # same read call as the repo's loader (it actually yields BGR; training used the same)
    rgb = cv2.imread(os.path.join(DATA, 'RGB', n + '.png'), cv2.COLOR_BGR2RGB).astype(np.float64)
    th = cv2.imread(os.path.join(DATA, 'Thermal', n + '.png'), cv2.IMREAD_GRAYSCALE)
    th = cv2.merge([th, th, th]).astype(np.float64)
    gt = cv2.imread(os.path.join(DATA, 'Label', n + '.png'), cv2.IMREAD_GRAYSCALE)
    pred = predict(prep(rgb), prep(th))
    keep = gt < 5
    cm += np.bincount(gt[keep].astype(int) * 5 + pred[keep], minlength=25).reshape(5, 5)
    if i % 50 == 0: print(f'{i}/{len(names)}')

tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; tn = cm.sum() - tp - fp - fn
res = pd.DataFrame({'IoU': tp / (tp + fp + fn), 'Dice': 2*tp / (2*tp + fp + fn),
                    'Sensitivity': tp / (tp + fn), 'Specificity': tn / (tn + fp)}, index=CLASSES) * 100
res.loc['Mean'] = res.mean()
res.loc['Mean (objects, no bg)'] = res.iloc[1:5].mean()
res = res.round(2)
res.to_csv(os.path.join(WORK, 'results.csv'))
torch.save({'model': model.state_dict()}, os.path.join(WORK, 'cmx_pst900_final_weights.pth'))
print(f'\nmIoU = {res.loc["Mean", "IoU"]:.2f}%')
res